# 03 — Gold Layer: Analytical Aggregations

This notebook demonstrates the **Gold layer** — the consumption-ready analytical tables.

**Tables produced:**
| Table | Grain | Description |
|---|---|---|
| `daily_metrics` | ticker × trade_date | OHLCV + rolling volatility, avg volume, cumulative return |
| `portfolio_summary` | ticker | Period totals: total return, avg volume, avg volatility |
| `monthly_returns` | ticker × year × month | Monthly OHLC and return |

In [11]:
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
pl.Config.set_tbl_rows(20)

polars.config.Config

## 1. Build Gold tables

In [12]:
from f_pipelines.gold_pipeline import GoldPipeline

tables = GoldPipeline().run()

for name, df in tables.items():
    print(f"{name:25s}: {len(df):>6,} rows  |  cols: {df.columns}")

{"timestamp": "2026-06-25T17:02:36.014205+00:00", "level": "INFO", "logger": "f_pipelines.gold_pipeline", "message": "GoldPipeline started", "module": "gold_pipeline", "func": "run", "line": 60, "name": "f_pipelines.gold_pipeline", "msg": "GoldPipeline started", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\f_pipelines\\gold_pipeline.py", "filename": "gold_pipeline.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 60, "funcName": "run", "created": 1782406956.0141633, "msecs": 14.0, "relativeCreated": 524691.9041, "thread": 30356, "threadName": "MainThread", "processName": "MainProcess", "process": 28544, "taskName": "Task-104"}
{"timestamp": "2026-06-25T17:02:36.015131+00:00", "level": "INFO", "logger": "f_pipelines.gold_pipeline", "message": "Reading Silver data for Gold pipeline", "module": "gold_pipeline", "func": "extract", "line": 41, "name": "f_pipelin

## 2. Daily metrics overview

In [3]:
daily = tables.get("daily_metrics")
if daily is not None:
    daily.head(10)

In [ ]:
# Comparativo de retornos acumulados
if daily is not None:
    top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
    plot_df = daily.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

    fig = px.line(
        plot_df,
        x="trade_date",
        y="cum_return",
        color="ticker",
        title="Retorno Acumulado por Ativo — Camada Gold",
        labels={
            "trade_date": "Data do Pregão",
            "cum_return": "Retorno Acumulado",
            "ticker": "Ativo",
        },
    )
    fig.add_hline(y=0, line_dash="dash", line_color="grey")
    fig.update_layout(
        height=520,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
        margin=dict(l=60, r=40, t=70, b=50),
    )
    fig.update_yaxes(tickformat=".1%")
    fig.update_xaxes(tickformat="%d/%m/%Y")
    fig.show()

In [ ]:
# Volatilidade anualizada (janela móvel de 20 dias)
if daily is not None:
    fig2 = px.line(
        plot_df,
        x="trade_date",
        y="volatility_20d",
        color="ticker",
        title="Volatilidade Anualizada — Janela Móvel de 20 dias",
        labels={
            "trade_date": "Data do Pregão",
            "volatility_20d": "Volatilidade Anualizada",
            "ticker": "Ativo",
        },
    )
    fig2.update_layout(
        height=520,
        hovermode="x unified",
        template="plotly_white",
        legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
        margin=dict(l=60, r=40, t=70, b=50),
    )
    fig2.update_yaxes(tickformat=".1%")
    fig2.update_xaxes(tickformat="%d/%m/%Y")
    fig2.show()

## 3. Portfolio summary

In [6]:
summary = tables.get("portfolio_summary")
if summary is not None:
    print("Top performers:")
    display(summary.head(10).to_pandas())

Top performers:


,ticker,first_date,last_date,total_return,avg_daily_volume,avg_volatility,period_high,period_low
0,VALE3.SA,2026-03-27,2026-06-24,1.359229e+27,1.887998e+07,46.304079,89.750000,77.699997
1,BPAC11.SA,2026-03-27,2026-06-24,2.106187e+15,1.023034e+07,27.456035,64.339996,49.200001
2,WEGE3.SA,2026-03-27,2026-06-24,3.899488e+14,8.147952e+06,29.339834,52.880001,41.779999
3,PETR4.SA,2026-03-27,2026-06-24,1.276866e+13,4.963668e+07,25.895724,49.779999,38.290001
4,RENT3.SA,2026-03-27,2026-06-24,5.950909e+10,9.488833e+06,19.255863,51.980000,39.119999
5,ITUB4.SA,2026-03-27,2026-06-24,2.848125e+10,2.964894e+07,25.299708,47.040001,38.520000
6,RADL3.SA,2026-03-27,2026-06-24,-9.999997e-01,1.108324e+07,9.087885,24.190001,16.250000
7,BBAS3.SA,2026-03-27,2026-06-24,-1.000000e+00,2.368228e+07,8.185624,25.379999,19.000000
8,BBDC4.SA,2026-03-27,2026-06-24,-1.000000e+00,3.075411e+07,8.990587,21.260000,17.200001
9,LREN3.SA,2026-03-27,2026-06-24,-1.000000e+00,1.663205e+07,9.175994,15.910000,13.140000


In [ ]:
# Dispersão Risco x Retorno
if summary is not None:
    sum_pd = summary.to_pandas()
    fig3 = px.scatter(
        sum_pd.dropna(subset=["total_return", "avg_volatility"]),
        x="avg_volatility",
        y="total_return",
        text="ticker",
        title="Risco x Retorno no Período (por Ativo)",
        labels={
            "avg_volatility": "Volatilidade Média (anualizada)",
            "total_return": "Retorno Total no Período",
            "ticker": "Ativo",
        },
        color="ticker",
    )
    fig3.update_traces(textposition="top center", marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))
    fig3.update_layout(
        height=560,
        template="plotly_white",
        showlegend=False,
        margin=dict(l=60, r=40, t=70, b=60),
    )
    fig3.add_hline(y=0, line_dash="dash", line_color="grey")
    fig3.update_xaxes(tickformat=".1%")
    fig3.update_yaxes(tickformat=".1%")
    fig3.show()

## 4. Monthly returns heatmap

In [8]:
monthly = tables.get("monthly_returns")
if monthly is not None:
    monthly.head(10)

In [ ]:
# Heatmap: ativo x mês -> retorno mensal
import seaborn as sns
import matplotlib.pyplot as plt

if monthly is not None:
    pivot = (
        monthly
        .filter(pl.col("ticker").is_in(top5))
        .with_columns(
            pl.concat_str(
                [pl.col("year").cast(pl.Utf8), pl.lit("-"), pl.col("month").cast(pl.Utf8).str.zfill(2)]
            ).alias("ym")
        )
        .pivot(values="month_return", index="ticker", on="ym")
        .to_pandas()
        .set_index("ticker")
        .astype(float)
    )

    fig4, ax = plt.subplots(figsize=(max(10, 1.1 * pivot.shape[1]), 4.5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1%",
        cmap="RdYlGn",
        center=0,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Retorno Mensal"},
        ax=ax,
    )
    ax.set_title("Mapa de Calor — Retorno Mensal por Ativo", fontsize=13, pad=12)
    ax.set_xlabel("Ano-Mês")
    ax.set_ylabel("Ativo")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Read persisted Gold tables

In [10]:
from d_processing.gold.aggregate import read_gold

gold_daily = read_gold("daily_metrics")
print(f"Persisted daily_metrics rows: {len(gold_daily):,}")

{"timestamp": "2026-06-25T16:56:09.657994+00:00", "level": "INFO", "logger": "d_processing.gold.aggregate", "message": "Gold read complete", "module": "aggregate", "func": "read_gold", "line": 163, "name": "d_processing.gold.aggregate", "msg": "Gold read complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\gold\\aggregate.py", "filename": "aggregate.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 163, "funcName": "read_gold", "created": 1782406569.6579385, "msecs": 657.0, "relativeCreated": 138335.6794, "thread": 30356, "threadName": "MainThread", "processName": "MainProcess", "process": 28544, "taskName": "Task-98", "table": "daily_metrics", "rows": 720}
Persisted daily_metrics rows: 720
